# 🧪 Taller Práctico: Transformers, BERT y RoBERTa

Este notebook acompaña una clase práctica de PLN sobre modelos de lenguaje basados en Transformers. Usaremos modelos preentrenados como BERT y RoBERTa para realizar tareas de análisis de sentimientos, clasificación de texto y análisis de representaciones.

## ✅ Parte 1: Uso de `pipeline` para tareas básicas de NLP

In [ ]:
from transformers import pipeline

# Clasificación de sentimiento
task = pipeline("sentiment-analysis")
text = "BERT es un modelo muy poderoso para NLP."
result = task(text)
print(result)

## ✅ Parte 2: Tokenización y extracción de embeddings

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "Transformers are powerful models for NLP."
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

print("Shape del embedding:", outputs.last_hidden_state.shape)

## ✅ Parte 3: Clasificación de texto con RoBERTa y dataset AG News

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score

# Cargar dataset AG News
dataset = load_dataset("ag_news")
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

def preprocess(example):
    return tokenizer(example["text"], truncation=True)

tokenized_ds = dataset.map(preprocess, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"].shuffle(seed=42).select(range(1000)),
    eval_dataset=tokenized_ds["test"].select(range(200)),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
eval_result = trainer.evaluate()
print(eval_result)

## ✅ Parte 4: Adaptación de RoBERTa en español para clasificación

En esta sección usaremos el modelo `PlanTL-GOB-ES/roberta-base-bne` (RoBERTa entrenado en corpus en español) y un dataset en español para clasificar sentimientos de tweets.

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification

# Dataset de ejemplo en español
es_dataset = load_dataset("pysentimiento/tweets-sentiment", split="train[:2000]")

model_name = "PlanTL-GOB-ES/roberta-base-bne"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def preprocess_es(example):
    return tokenizer(example['text'], truncation=True)

encoded_dataset = es_dataset.map(preprocess_es, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Métricas
import numpy as np

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="no",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()